In [ ]:
# Cell 1 — Parameters
TENOR = '10Y'
COUNTRIES = {
    'Peru': df_perugb_cmt,
    'Mexico': df_mbono_cmt,
    'Colombia': df_coltes_cmt,
    'Chile': df_btpcl_cmt,
}
FOCUS_COUNTRY = 'Peru'
LOOKBACK_START = '2022-01-01'  # None = full history
N_PCS = 2  # None = auto via 90% cumvar
SD_WINDOW = 252
SD_BANDS = [1.25, 1.65]
ROLLING_WINDOWS = [5, 10, 20]
ROLL_DISPLAY = [5, 20]
TRAIL_WINDOW = 60

In [ ]:
# Cell 2 — Build Spread Panel
import pandas as pd
import numpy as np

def build_spread_panel(countries, tenor, ust_df):
    ust = ust_df.set_index('Fecha')[tenor].rename('UST')
    frames = {}
    for name, df in countries.items():
        s = df.set_index('Fecha')[tenor]
        aligned = pd.concat([s, ust], axis=1).dropna()
        frames[name] = (aligned[tenor] - aligned['UST']) * 100
    df_out = pd.DataFrame(frames)
    df_out.index.name = 'Fecha'
    return df_out

df_spreads = build_spread_panel(COUNTRIES, TENOR, df_ust_cmt)
print(f'Shape: {df_spreads.shape}')
print(f'Date range: {df_spreads.index.min().date()} — {df_spreads.index.max().date()}')

In [ ]:
# Cell 3 — PCA Engine + Decompose

def run_pca(df_spreads, lookback_start, n_pcs):
    df = df_spreads.copy()
    if lookback_start is not None:
        df = df[df.index >= lookback_start]
    df = df.dropna()
    means = df.mean()
    demeaned = df - means
    cov = demeaned.cov().values
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    # sort descending
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]
    var_explained = eigenvalues / eigenvalues.sum()
    cum_var = np.cumsum(var_explained)
    if n_pcs is None:
        n_pcs = int(np.searchsorted(cum_var, 0.90)) + 1
    loadings = pd.DataFrame(
        eigenvectors[:, :n_pcs],
        index=df.columns,
        columns=[f'PC{i+1}' for i in range(n_pcs)]
    )
    scores = pd.DataFrame(
        demeaned.values @ eigenvectors[:, :n_pcs],
        index=df.index,
        columns=[f'PC{i+1}' for i in range(n_pcs)]
    )
    return {
        'means': means,
        'loadings': loadings,
        'scores': scores,
        'eigenvalues': eigenvalues[:n_pcs],
        'var_explained': var_explained[:n_pcs],
        'cum_var_explained': cum_var[:n_pcs],
        'n_pcs': n_pcs,
    }

def decompose(pca, df_spreads, lookback_start=None):
    df = df_spreads.copy()
    if lookback_start is not None:
        df = df[df.index >= lookback_start]
    df = df.dropna()
    demeaned = df - pca['means']
    demeaned = demeaned.dropna()
    loadings = pca['loadings']
    scores = pd.DataFrame(
        demeaned.values @ loadings.values,
        index=demeaned.index,
        columns=loadings.columns
    )
    fitted = pd.DataFrame(
        scores.values @ loadings.values.T + pca['means'].values,
        index=demeaned.index,
        columns=df.columns
    )
    residuals = df.loc[demeaned.index] - fitted
    contributions = {}
    for country in df.columns:
        contrib = pd.DataFrame(index=demeaned.index)
        for pc in loadings.columns:
            contrib[pc] = scores[pc] * loadings.loc[country, pc]
        contributions[country] = contrib
    return {
        'fitted': fitted,
        'residuals': residuals,
        'contributions': contributions,
        'scores_full': scores,
    }

pca = run_pca(df_spreads, LOOKBACK_START, N_PCS)
decomp = decompose(pca, df_spreads, LOOKBACK_START)

print(f'PCs retained: {pca["n_pcs"]}')
print(f'Variance explained: {[f"{v:.1%}" for v in pca["var_explained"]]}')
print(f'Fitted date range: {decomp["fitted"].index.min().date()} — {decomp["fitted"].index.max().date()}')
print(f'Residuals shape: {decomp["residuals"].shape}')

In [ ]:
# Cell 4 — Country Selection Diagnostic
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

countries_all = {
    'Peru': df_perugb_cmt,
    'Mexico': df_mbono_cmt,
    'Colombia': df_coltes_cmt,
    'Chile': df_btpcl_cmt,
}
countries_ex_mexico = {
    'Peru': df_perugb_cmt,
    'Colombia': df_coltes_cmt,
    'Chile': df_btpcl_cmt,
}

sp_all = build_spread_panel(countries_all, TENOR, df_ust_cmt)
sp_exmx = build_spread_panel(countries_ex_mexico, TENOR, df_ust_cmt)

pca_all = run_pca(sp_all, LOOKBACK_START, N_PCS)
pca_exmx = run_pca(sp_exmx, LOOKBACK_START, N_PCS)

# side-by-side variance and loadings
print('=== All Countries ===')
for i, (v, cv) in enumerate(zip(pca_all['var_explained'], pca_all['cum_var_explained'])):
    print(f'  PC{i+1}: {v:.1%}  cumulative: {cv:.1%}')
print(pca_all['loadings'].to_string(float_format=lambda x: f'{x:.4f}'))

print('\n=== Ex-Mexico ===')
for i, (v, cv) in enumerate(zip(pca_exmx['var_explained'], pca_exmx['cum_var_explained'])):
    print(f'  PC{i+1}: {v:.1%}  cumulative: {cv:.1%}')
print(pca_exmx['loadings'].to_string(float_format=lambda x: f'{x:.4f}'))

# Peru residual comparison
decomp_all = decompose(pca_all, sp_all, LOOKBACK_START)
decomp_exmx = decompose(pca_exmx, sp_exmx, LOOKBACK_START)

for label, d in [('All Countries', decomp_all), ('Ex-Mexico', decomp_exmx)]:
    r = d['residuals']['Peru']
    ac1 = r.autocorr(1)
    ac5 = r.autocorr(5)
    print(f'\nPeru residuals [{label}]: Std={r.std():.2f}  AC(1)={ac1:.3f}  AC(5)={ac5:.3f}  Min={r.min():.1f}  Max={r.max():.1f}')

# grouped bar charts
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, label, pca_r in [(axes[0], 'All Countries', pca_all), (axes[1], 'Ex-Mexico', pca_exmx)]:
    ld = pca_r['loadings']
    n_countries = len(ld)
    x = np.arange(n_countries)
    n_pcs_r = pca_r['n_pcs']
    width = 0.8 / n_pcs_r
    colors = plt.cm.tab10.colors
    for i, pc in enumerate(ld.columns):
        vals = ld[pc].values
        bars = ax.bar(x + i * width - (n_pcs_r - 1) * width / 2, vals, width, label=pc, color=colors[i])
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005 * np.sign(val + 1e-9),
                    f'{val:.2f}', ha='center', va='bottom' if val >= 0 else 'top', fontsize=8)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(ld.index, rotation=15)
    ax.set_title(f'PCA Loadings — {label}')
    ax.legend()

plt.suptitle(f'Country Selection Diagnostic ({TENOR})', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 5 — Factor Returns & Rolling Sums
factor_returns = decomp['scores_full'].diff().dropna()
roll_factor = {}
for w in ROLLING_WINDOWS:
    roll_factor[w] = factor_returns.rolling(w).sum()

print(f'Factor returns shape: {factor_returns.shape}')
print(f'Rolling windows computed: {ROLLING_WINDOWS}')

In [ ]:
# Cell 6 — Variance Explained Table + Bar Chart
import matplotlib.pyplot as plt

# table
print(f'{"PC":<6}{"Var Explained":>15}{"Cum Var Explained":>20}')
print('-' * 42)
for i, (v, cv) in enumerate(zip(pca['var_explained'], pca['cum_var_explained'])):
    print(f'PC{i+1:<4}{v*100:>14.2f}%{cv*100:>19.2f}%')

# bar chart
fig, ax = plt.subplots(figsize=(7, 4))
pcs = [f'PC{i+1}' for i in range(pca['n_pcs'])]
vals = pca['var_explained'] * 100
bars = ax.bar(pcs, vals, color='steelblue')
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=10)
ax.set_ylabel('Variance Explained (%)')
ax.set_title(f'PCA Variance Explained ({TENOR})')
plt.tight_layout()
plt.show()

# loadings matrix
print('\nLoadings:')
print(pca['loadings'].to_string(float_format=lambda x: f'{x:.4f}'))

In [ ]:
# Cell 7 — Loadings Bar Chart
fig, ax = plt.subplots(figsize=(9, 5))
ld = pca['loadings']
n_countries = len(ld)
x = np.arange(n_countries)
n_pcs_plot = pca['n_pcs']
width = 0.8 / n_pcs_plot
colors = plt.cm.tab10.colors

for i, pc in enumerate(ld.columns):
    vals = ld[pc].values
    bars = ax.bar(x + i * width - (n_pcs_plot - 1) * width / 2, vals, width,
                  label=pc, color=colors[i])
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.005 * (1 if val >= 0 else -1),
                f'{val:.2f}', ha='center',
                va='bottom' if val >= 0 else 'top', fontsize=9)

ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(ld.index, rotation=15)
ax.set_ylabel('Loading')
ax.set_title(f'PCA Loadings by Country ({TENOR})')
ax.legend(title='PC')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 8 — Loading Magnitude vs Volatility
df_lb = df_spreads.copy()
if LOOKBACK_START is not None:
    df_lb = df_lb[df_lb.index >= LOOKBACK_START]
df_lb = df_lb.dropna()

daily_chg_sd = df_lb.diff().std()
pc1_abs = pca['loadings']['PC1'].abs()

# normalize to largest country
max_sd = daily_chg_sd.max()
max_loading = pc1_abs.max()
vol_ratio = daily_chg_sd / max_sd
loading_ratio = pc1_abs / max_loading
ratio_match = vol_ratio / loading_ratio

print(f'{"Country":<12}{"Daily Chg SD":>14}{"|PC1 Loading|":>15}{"Vol Ratio":>12}{"Loading Ratio":>15}{"Ratio Match":>13}')
print('-' * 82)
for country in df_lb.columns:
    print(f'{country:<12}{daily_chg_sd[country]:>13.3f} '
          f'{pc1_abs[country]:>14.4f} '
          f'{vol_ratio[country]:>11.4f} '
          f'{loading_ratio[country]:>14.4f} '
          f'{ratio_match[country]:>12.4f}')

print('\nRatio Match near 1.0 confirms loadings are volatility-driven.')

In [ ]:
# Cell 9 — Correlation-Based PCA Diagnostic

def run_pca_corr(df_spreads, lookback_start, n_pcs):
    df = df_spreads.copy()
    if lookback_start is not None:
        df = df[df.index >= lookback_start]
    df = df.dropna()
    standardized = (df - df.mean()) / df.std()
    cov = standardized.cov().values
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]
    var_explained = eigenvalues / eigenvalues.sum()
    cum_var = np.cumsum(var_explained)
    if n_pcs is None:
        n_pcs = int(np.searchsorted(cum_var, 0.90)) + 1
    loadings = pd.DataFrame(
        eigenvectors[:, :n_pcs],
        index=df.columns,
        columns=[f'PC{i+1}' for i in range(n_pcs)]
    )
    return {
        'loadings': loadings,
        'var_explained': var_explained[:n_pcs],
        'n_pcs': n_pcs,
    }

pca_corr = run_pca_corr(df_spreads, LOOKBACK_START, N_PCS)

# side-by-side variance explained
print(f'{"PC":<6}{"Cov Var%":>10}{"Corr Var%":>12}')
print('-' * 30)
for i in range(max(pca['n_pcs'], pca_corr['n_pcs'])):
    cv = pca['var_explained'][i] * 100 if i < pca['n_pcs'] else float('nan')
    cc = pca_corr['var_explained'][i] * 100 if i < pca_corr['n_pcs'] else float('nan')
    print(f'PC{i+1:<4}{cv:>9.2f}%{cc:>11.2f}%')

# loadings side by side
print('\nCovariance Loadings:')
print(pca['loadings'].to_string(float_format=lambda x: f'{x:.4f}'))
print('\nCorrelation Loadings:')
print(pca_corr['loadings'].to_string(float_format=lambda x: f'{x:.4f}'))

# two grouped bar charts
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, label, pca_r in [(axes[0], 'Covariance', pca), (axes[1], 'Correlation', pca_corr)]:
    ld = pca_r['loadings']
    n_c = len(ld)
    x = np.arange(n_c)
    n_pcs_r = pca_r['n_pcs']
    width = 0.8 / n_pcs_r
    colors = plt.cm.tab10.colors
    for i, pc in enumerate(ld.columns):
        vals = ld[pc].values
        bars = ax.bar(x + i * width - (n_pcs_r - 1) * width / 2, vals, width,
                      label=pc, color=colors[i])
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.005 * (1 if val >= 0 else -1),
                    f'{val:.2f}', ha='center',
                    va='bottom' if val >= 0 else 'top', fontsize=8)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(ld.index, rotation=15)
    ax.set_title(f'{label}-Based Loadings')
    ax.legend()

plt.suptitle(f'PCA Loadings — Covariance vs Correlation ({TENOR})', fontsize=13)
plt.tight_layout()
plt.show()
print('Note: Correlation loadings are more balanced (volatility removed). Covariance PCA remains the working model.')

In [ ]:
# Cell 10 — Main Diagnostic Chart (FOCUS_COUNTRY)
import matplotlib.patches as mpatches

def shade_bands(ax, dates, series, roll_sd, bands):
    # shade between inner and outer and beyond outer
    inner, outer = bands[0], bands[1]
    for i in range(len(dates) - 1):
        v = series.iloc[i]
        sd = roll_sd.iloc[i]
        if pd.isna(v) or pd.isna(sd) or sd == 0:
            continue
        z = v / sd
        x0, x1 = dates[i], dates[i + 1]
        if z > outer:
            ax.axvspan(x0, x1, color='red', alpha=0.3, linewidth=0)
        elif z > inner:
            ax.axvspan(x0, x1, color='orange', alpha=0.2, linewidth=0)
        elif z < -outer:
            ax.axvspan(x0, x1, color='darkblue', alpha=0.3, linewidth=0)
        elif z < -inner:
            ax.axvspan(x0, x1, color='lightblue', alpha=0.4, linewidth=0)

country = FOCUS_COUNTRY
actual = df_spreads[country].dropna()
fitted = decomp['fitted'][country]
residual = decomp['residuals'][country]
contrib = decomp['contributions'][country]
mean_val = pca['means'][country]
roll_sd = residual.rolling(SD_WINDOW).std()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

# top panel: stacked area of contributions + mean as base
pcs = contrib.columns.tolist()
colors_area = plt.cm.tab10.colors
base = pd.Series(mean_val, index=contrib.index)
bottom_pos = base.copy()
bottom_neg = base.copy()
ax1.fill_between(contrib.index, 0, base, alpha=0.15, color='gray', label=f'Mean ({mean_val:.1f} bps)')
for i, pc in enumerate(pcs):
    vals = contrib[pc]
    upper_pos = bottom_pos + vals.clip(lower=0)
    upper_neg = bottom_neg + vals.clip(upper=0)
    ax1.fill_between(contrib.index, bottom_pos, upper_pos, alpha=0.5,
                     color=colors_area[i], label=pc)
    ax1.fill_between(contrib.index, bottom_neg, upper_neg, alpha=0.5,
                     color=colors_area[i])
    bottom_pos = upper_pos
    bottom_neg = upper_neg
ax1.plot(actual.index, actual, color='black', linewidth=1.5, label='Actual')
ax1.set_ylabel('Spread (bps)')
ax1.legend(loc='upper left', fontsize=8)
ax1.set_title(
    f'{FOCUS_COUNTRY} | {TENOR} Spread vs UST | '
    f'PCA from {LOOKBACK_START if LOOKBACK_START else "full history"}'
)

# bottom panel: residual + SD bands + shading
dates = residual.index.tolist()
shade_bands(ax2, dates, residual, roll_sd, SD_BANDS)
ax2.plot(residual.index, residual, color='black', linewidth=1.2, label='Residual')
ax2.axhline(0, color='black', linewidth=0.6)
band_colors = ['darkorange', 'red']
for thresh, col in zip(SD_BANDS, band_colors):
    ub = roll_sd * thresh
    lb = -roll_sd * thresh
    ax2.plot(roll_sd.index, ub, linestyle='--', color=col, linewidth=0.9,
             label=f'+{thresh}σ')
    ax2.plot(roll_sd.index, lb, linestyle='--', color=col, linewidth=0.9,
             label=f'-{thresh}σ')
ax2.set_ylabel('Residual (bps)')
ax2.legend(loc='upper left', fontsize=8, ncol=3)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 11 — All Countries Residual Dashboard
countries_list = list(decomp['residuals'].columns)
n_c = len(countries_list)
fig, axes = plt.subplots(n_c, 1, figsize=(14, 4 * n_c), sharex=True)
if n_c == 1:
    axes = [axes]

band_colors = ['darkorange', 'red']

for ax, cname in zip(axes, countries_list):
    resid = decomp['residuals'][cname]
    roll_sd = resid.rolling(SD_WINDOW).std()
    dates = resid.index.tolist()
    shade_bands(ax, dates, resid, roll_sd, SD_BANDS)
    ax.plot(resid.index, resid, color='black', linewidth=1.1)
    ax.axhline(0, color='black', linewidth=0.6)
    for thresh, col in zip(SD_BANDS, band_colors):
        ax.plot(roll_sd.index, roll_sd * thresh, linestyle='--', color=col, linewidth=0.8)
        ax.plot(roll_sd.index, -roll_sd * thresh, linestyle='--', color=col, linewidth=0.8)
    ax.set_ylabel('Residual (bps)')
    ax.set_title(cname)

plt.suptitle(f'All Countries — {TENOR} Spread Residuals vs UST', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 12 — PC Scores with Regime Shading
scores = decomp['scores_full']
n_pcs_plot = pca['n_pcs']
fig, axes = plt.subplots(n_pcs_plot, 1, figsize=(14, 4 * n_pcs_plot), sharex=True)
if n_pcs_plot == 1:
    axes = [axes]

for ax, pc in zip(axes, scores.columns):
    s = scores[pc]
    trail = s.rolling(TRAIL_WINDOW).mean()
    # regime shading
    for i in range(len(s) - 1):
        if pd.isna(trail.iloc[i]):
            continue
        x0, x1 = s.index[i], s.index[i + 1]
        if s.iloc[i] > trail.iloc[i]:
            ax.axvspan(x0, x1, color='green', alpha=0.15, linewidth=0)
        else:
            ax.axvspan(x0, x1, color='red', alpha=0.15, linewidth=0)
    ax.plot(s.index, s, color='steelblue', linewidth=1.2, label=pc)
    ax.plot(trail.index, trail, color='navy', linestyle='--', linewidth=1.0,
            label=f'{TRAIL_WINDOW}d trailing mean')
    ax.axhline(0, color='black', linewidth=0.6)
    ax.set_ylabel('Score')
    ax.set_title(pc)
    ax.legend(fontsize=8)

plt.suptitle(f'PC Scores — Levels and Regime ({TENOR})', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 13 — Factor Return Momentum
n_pcs_plot = pca['n_pcs']
fig, axes = plt.subplots(n_pcs_plot, 1, figsize=(14, 4 * n_pcs_plot), sharex=True)
if n_pcs_plot == 1:
    axes = [axes]

roll_colors = plt.cm.tab10.colors

for ax, pc in zip(axes, factor_returns.columns):
    for j, w in enumerate(ROLL_DISPLAY):
        ax.plot(roll_factor[w].index, roll_factor[w][pc],
                color=roll_colors[j], linewidth=1.2, label=f'{w}d')
    ax.axhline(0, color='black', linewidth=0.6)
    ax.set_ylabel('Rolling Sum')
    ax.set_title(pc)
    ax.legend(fontsize=8)

plt.suptitle(f'Factor Return Momentum — Rolling Sums ({TENOR})', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 14 — Current Snapshot Table
last_date = decomp['residuals'].index[-1]
resid_last = decomp['residuals'].loc[last_date]
actual_last = df_spreads.loc[last_date] if last_date in df_spreads.index else decomp['fitted'].loc[last_date] + resid_last
fitted_last = decomp['fitted'].loc[last_date]

# rolling SD for z-score
roll_sd_all = {c: decomp['residuals'][c].rolling(SD_WINDOW).std() for c in decomp['residuals'].columns}

pcs = pca['loadings'].columns.tolist()

print(f'=== Snapshot as of {last_date.date()} | Tenor: {TENOR} ===\n')

# header
pc_headers = '  '.join([f'{pc:>10}' for pc in pcs])
print(f'{"Country":<12}{"Actual":>10}{"Fitted":>10}{"Residual":>11}{"Z-Score":>10}  {pc_headers}')
print('-' * (12 + 10 + 10 + 11 + 10 + 2 + 12 * len(pcs)))

for country in decomp['residuals'].columns:
    act = actual_last[country] if country in actual_last.index else float('nan')
    fit = fitted_last[country]
    res = resid_last[country]
    sd_val = roll_sd_all[country].iloc[-1]
    z = res / sd_val if not pd.isna(sd_val) and sd_val != 0 else float('nan')
    contrib_last = decomp['contributions'][country].loc[last_date]
    pc_vals = '  '.join([f'{contrib_last[pc]:>10.1f}' for pc in pcs])
    print(f'{country:<12}{act:>10.1f}{fit:>10.1f}{res:>11.1f}{z:>10.2f}  {pc_vals}')